# Experiment A / A2: Zero-Copy Data Plane Charts

Tweakable notebook. Each chart's plotting code is inline and self-contained — edit colors, labels, scales, and figure sizes here, then re-run the cell. Run this notebook from its own directory so the relative paths resolve.

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams.update({
    'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 12,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 10,
    'figure.figsize': (10, 6),
    # Match the quals scaling notebook: keep the box frame, no gridlines,
    # white background, black-edged bars, legend in a bordered box below.
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'savefig.facecolor': 'white', 'axes.grid': False,
})
BAR_EDGE, BAR_LW = 'black', 0.8

# Semantic colors: proposed system green, naive text red, zero-copy binary orange.
FORMAT_COLORS = {
    'arrow_ipc': '#2ca02c', 'contexttable_ipc': '#17becf', 'flatbuffers': '#ff7f0e',
    'pickle': '#7f7f7f', 'protobuf': '#8c564b', 'base64_tobytes': '#9467bd', 'json': '#d62728',
}
# Display order (zero-copy formats grouped first). Reorder to taste.
FORMAT_ORDER = ['arrow_ipc', 'contexttable_ipc', 'flatbuffers', 'pickle',
                'base64_tobytes', 'protobuf', 'json']


In [ ]:
df = pd.read_csv('exp_a_zerocopy_results.csv')
a2 = pd.read_csv('exp_a2_schema_compat.csv')
# Representative (largest) cell for the bar charts.
N_REP = int(df['n'].max())
D_REP = int(df[df['n'] == N_REP]['d'].max())
cell = df[(df['n'] == N_REP) & (df['d'] == D_REP)].set_index('format')
fmts = [f for f in FORMAT_ORDER if f in cell.index]
print(f"formats: {list(df['format'].unique())}")
print(f"representative cell: N={N_REP}, D={D_REP}")
display(df.head(10))


## Throughput by format (log scale)

In [ ]:
vals = [cell.loc[f, 'throughput_roundtrip_MBps'] for f in fmts]
sp = [cell.loc[f, 'speedup_vs_json'] for f in fmts]
colors = [FORMAT_COLORS.get(f, '#333') for f in fmts]

fig, ax = plt.subplots()
y = range(len(fmts))
# Bar chart is a representative-cell snapshot — no error bars (per convention).
ax.barh(list(y), vals, color=colors, alpha=0.9, edgecolor=BAR_EDGE, linewidth=BAR_LW)
ax.set_yticks(list(y)); ax.set_yticklabels(fmts); ax.invert_yaxis()
ax.set_xscale('log'); ax.minorticks_off()
ax.set_xlabel('Round-trip throughput (MB/s, log scale)')
ax.set_title(f'Zero-copy data plane throughput by format  (N={N_REP}, D={D_REP})')
for i, (v, s) in enumerate(zip(vals, sp)):
    lbl = f'{v:,.0f} MB/s' + (f'  ({s:,.0f}x vs JSON)' if pd.notna(s) else '')
    ax.text(v * 1.05, i, lbl, va='center', fontsize=9)
ax.set_xlim(right=max(vals) * 12)
plt.tight_layout(); plt.savefig('charts/exp_a_throughput_by_format.svg', bbox_inches='tight'); plt.show()


## Memory copies on read

In [ ]:
peaks = [max(cell.loc[f, 'deserialize_peak_kb'], 0.1) for f in fmts]
copies = [cell.loc[f, 'memory_copies'] for f in fmts]
colors = [FORMAT_COLORS.get(f, '#333') for f in fmts]
payload_kb = N_REP * D_REP * 4 / 1024

fig, ax = plt.subplots()
x = range(len(fmts))
ax.bar(list(x), peaks, color=colors, alpha=0.9, edgecolor=BAR_EDGE, linewidth=BAR_LW); ax.set_yscale('log'); ax.minorticks_off()
ax.set_xticks(list(x)); ax.set_xticklabels(fmts, rotation=30, ha='right')
ax.set_ylabel('Deserialize peak allocation (KB, log scale)')
ax.set_title(f'Memory copies on read  (N={N_REP}, D={D_REP}; payload = {payload_kb:,.0f} KB)')
for i, (p, c) in enumerate(zip(peaks, copies)):
    ax.text(i, p * 1.3, f"{int(c)} cop{'y' if int(c) == 1 else 'ies'}", ha='center', fontsize=9)
plt.tight_layout(); plt.savefig('charts/exp_a_memory_copies.svg', bbox_inches='tight'); plt.show()


## Throughput vs embedding dimension

In [ ]:
dims = sorted(df['d'].unique())
present = [f for f in FORMAT_ORDER if f in set(df['format'])]
STD = 'throughput_roundtrip_MBps_std'
fig, ax = plt.subplots()
for f in present:
    ys, es = [], []
    for dd in dims:
        cand = df[(df['format'] == f) & (df['d'] == dd)]
        if len(cand):
            top = cand.sort_values('n').iloc[-1]
            ys.append(top['throughput_roundtrip_MBps'])
            es.append(top[STD] if STD in df.columns else 0)
        else:
            ys.append(None); es.append(0)
    color = FORMAT_COLORS.get(f, '#333')
    yerr = es if any(e > 0 for e in es) else None
    ax.errorbar(dims, ys, yerr=yerr, marker='o', label=f, color=color, ecolor=color, lw=2, capsize=5)
ax.set_yscale('log'); ax.minorticks_off(); ax.set_xticks(dims)
ax.set_xlabel('Embedding dimension D'); ax.set_ylabel('Round-trip throughput (MB/s, log scale)')
ax.set_title('Throughput vs embedding dimension')
ax.legend(ncol=2, loc='lower center', bbox_to_anchor=(0.5, -0.32), frameon=True)
plt.tight_layout(); plt.savefig('charts/exp_a_throughput_scaling.svg', bbox_inches='tight'); plt.show()


## A2 schema compatibility matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(3, 0.9 * len(a2) + 1))); ax.axis('off')
tbl, colcolors = [], []
for _, e in a2.iterrows():
    zc = bool(e['zero_copy_eligible']); compat = bool(e['arrow_compatible'])
    status = 'zero-copy' if zc else ('compatible' if compat else 'needs serialization')
    tbl.append([e['edge'], e['payload_class'], status])
    colcolors.append('#2ca02c' if zc else ('#ff7f0e' if compat else '#d62728'))
t = ax.table(cellText=tbl, colLabels=['Operator edge', 'Payload class', 'Transfer'],
             loc='center', cellLoc='left')
t.auto_set_font_size(False); t.set_fontsize(12); t.scale(1, 1.8)
for i, c in enumerate(colcolors):
    t[i + 1, 2].set_facecolor(c); t[i + 1, 2].set_alpha(0.35)
ax.set_title('A2: Arrow schema compatibility across the operator DAG', pad=20)
plt.tight_layout(); plt.savefig('charts/exp_a2_schema_compat.svg', bbox_inches='tight'); plt.show()


## Summary table

In [ ]:
cols = ['format', 'n', 'd', 'wire_ratio', 'roundtrip_ms',
        'throughput_roundtrip_MBps', 'memory_copies', 'deserialize_peak_kb', 'speedup_vs_json']
available = [c for c in cols if c in df.columns]
display(df[available].sort_values(['d', 'throughput_roundtrip_MBps'], ascending=[True, False]))
